# Step 4 — Boundary surfaces

**Learning objective:** Identify and inspect the two top boundary meshes carrying Dirichlet conditions.

**Run from top to bottom in VS Code.** Each notebook executes one workshop step and shows its output. Select the working OGS Postprocessing Python kernel.

**Prerequisites:** run earlier numbered steps. `GMSH2OGS` and `ogs` must be on PATH when used; Gmsh is imported from the selected Python kernel in Step 2. **Every image below represents a distinct quantity.** Green PASS messages verify the stated condition only.

**Repeatable step:** The boundary assignment script reads the numbered GMSH2OGS submeshes (`excavation_1.vtu`, `excavation_2.vtu`, etc.) and regenerates the two named boundary meshes (`excavation_101.vtu`, `excavation_102.vtu`). Existing named outputs are ignored as inputs, so running this cell a second time does not produce duplicate-candidate errors.

In [ ]:
from pathlib import Path
import sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
import meshio
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents, Path.cwd().parent) if (p / "models" / "excavation.prj").exists()), None)
assert ROOT is not None, "Open the extracted project directory in VS Code before running this notebook"
print('Training root:', ROOT)
print('Python:', sys.executable)
def run(*command):
    command = [str(c) for c in command]
    print('RUN:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


**Run:** find GMSH2OGS-generated boundary segments geometrically, produce named boundary meshes 101 and 102.

In [ ]:
run(sys.executable,'scripts/assign_boundary_meshes.py')

### Terminal alternative — Step 4: extract the two top boundary surfaces

After terminal Steps 2 and 3, run the following in the course root:

```bash
python3 scripts/assign_boundary_meshes.py terminal_mesh_output
ls -lh terminal_mesh_output/excavation_101.vtu terminal_mesh_output/excavation_102.vtu
```

The script identifies the top-left and top-right GMSH2OGS line submeshes by geometry and saves the correctly numbered `excavation_101.vtu` and `excavation_102.vtu` (with `bulk_node_ids` mappings) under `terminal_mesh_output/`. The source numbered files are preserved. Re-running this step is safe. These independent command-line outputs **do not replace** the `mesh/` files referenced by `models/excavation.prj`.


**Inspect — 1D boundary submeshes, not material subdomains:** Red nodes mark the **upper-left Dirichlet boundary** (`excavation_101.vtu`, u = 60); blue nodes mark the **upper-right Dirichlet boundary** (`excavation_102.vtu`, u = 30). The underlying pale triangles are the bulk mesh (MaterialID 0). The narrow wall gap is excluded from the model: it is **not** one of these Dirichlet boundaries. The `bulk_node_ids` arrays tie boundary nodes to the global OGS degrees of freedom.

**PASS** checks geometric identity of every boundary node with the corresponding bulk node.

In [ ]:
bulk=meshio.read(ROOT/'mesh'/'excavation.vtu')
tri=np.vstack([b.data for b in bulk.cells if b.type=='triangle'])
fig,ax=plt.subplots(figsize=(12,5.8))
ax.triplot(bulk.points[:,0],bulk.points[:,1],tri,color='#cbd5e1',lw=.35)
for tag,color,label in [(101,'#e11d48','BC 101: u = 60'),(102,'#2563eb','BC 102: u = 30')]:
    b=meshio.read(ROOT/'mesh'/f'excavation_{tag}.vtu')
    mapping=b.point_data['bulk_node_ids'].reshape(-1).astype(int)
    assert np.allclose(b.points,bulk.points[mapping],atol=1e-7)
    ax.scatter(b.points[:,0],b.points[:,1],s=22,c=color,label=f'{label} ({len(b.points)} nodes)',zorder=5)
ax.legend(loc='lower center');ax.set(xlim=(0,65),ylim=(0,32),xlabel='x [m]',ylabel='y [m]',title='Boundary submeshes and bulk-node IDs')
ax.set_aspect('equal');plt.show()
print('PASS: boundary node mappings verified')

**Interpretation / Question:** The two coloured point sets are *where* OGS enforces fixed values. They do not represent two soil materials. **Next:** Step 5 connects these boundary submeshes to the values in `models/excavation.prj`.